# SMIC

**Benchmark Retrievability and Symmetry-Induced Memory Collapse in Sign Language Recognition**

Harapriya Kar, Sachit Raj Chaudhary, Saurya Pratap Singh, Sushma Viswanathan, Viswanathan P

This notebook reproduces the main benchmark of the paper (Table 2): five sequence models that share one convolutional trunk, evaluated on three sign language corpora under a leakage-controlled protocol. SMIC is the model proposed in the paper. It has no recurrence, and it is trained in two stages:

1. **Contrastive pretraining without labels.** The convolutional trunk learns to match two randomly augmented views of the same hand image (NT-Xent loss, temperature 0.2). For ISL-IEEE and ASL-IEEE it sees only the training split, with labels hidden. For CSL it sees frames of CSL signs that are not in the evaluation set.
2. **Supervised training.** The projection head is discarded, the trunk output tokens are averaged, layer-normalised and passed to a linear classifier, and the whole network is trained with cross-entropy.

The four comparison models use the same trunk and differ only in what reads the token sequence: mean pooling from random initialisation (CNN), a two-layer Transformer encoder, a two-layer LSTM, and the referenced STLAT dual-memory model with its adaptive fusion gate.

| Section | Content |
|---|---|
| 1 | Environment and configuration |
| 2 | Data and the lookup baseline |
| 3 | Models |
| 4 | Stage 1: contrastive pretraining |
| 5 | Stage 2: supervised training and evaluation |
| 6 | Results and statistical tests |
| 7 | Training curves |
| 8 | Proposition checks and unit tests |

Runtime: a GPU runtime is recommended (Runtime > Change runtime type > GPU). The full benchmark takes roughly one hour on a T4.

## 1. Environment and configuration

In [ ]:
import os
import sys
import json
import time
import random
import subprocess

IN_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/sachitrazz/SMIC.git'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/SMIC'
    REPO = '/content/SMIC'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO], check=True)
else:
    ROOT = os.environ.get('SMIC_ROOT', os.path.abspath('.'))
    REPO = os.environ.get('SMIC_REPO', os.path.abspath('.'))

DATA_DIR = os.path.join(ROOT, 'data')
CACHE_DIR = os.path.join(ROOT, 'cache')
RESULTS_DIR = os.path.join(ROOT, 'results')
for d in (CACHE_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, REPO)

print('data    ', DATA_DIR)
print('cache   ', CACHE_DIR)
print('results ', RESULTS_DIR)
print('code    ', REPO)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV,
      '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU')

The protocol below is identical for every model and corpus. All values are final-epoch values under a cosine learning-rate schedule; no epoch is selected on the validation set.

In [ ]:
RES = 64                                  # input resolution of the hand crops
WIDTH = 24                                # trunk widths 24, 48, 96, 96
SEEDS = [42, 123, 456, 789, 1024]         # shared by every model
EPOCHS = 40                               # supervised training
LR, WEIGHT_DECAY = 1e-3, 1e-3

PRETRAIN_EPOCHS = 20                      # contrastive stage
PRETRAIN_STEPS = 80                       # optimiser steps per pretraining epoch
PRETRAIN_BATCH = 128
TEMPERATURE = 0.2

CORPORA = ['isl', 'asl', 'csl']
FILES = {'isl': 'isl_grouped', 'asl': 'asl_grouped', 'csl': 'csl_signer'}
MODELS = ['cnn', 'cnn_tf', 'lstm', 'stlat', 'smic']
NAMES = {'cnn': 'CNN (mean-pooled tokens)', 'cnn_tf': 'CNN + Transformer',
         'lstm': 'CNN + LSTM', 'stlat': 'STLAT (dual memory + ASFG)',
         'smic': 'SMIC (CNN, contrastive init.)'}


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 2. Data and the lookup baseline

The corpora are not redistributed with the code. ISL-IEEE (doi:10.21227/796w-a432) and ASL-IEEE (doi:10.21227/4dz0-xv55) are available from IEEE DataPort, and CSL is available from its authors. The repository scripts `prepare_new.py`, `regroup_split.py` and `prepare_csl_signer.py` build the arrays used here from the raw corpora: exact duplicates are removed by MD5 hash, each image is cropped to the signing hand, near-duplicate groups are formed at cosine similarity 0.98, and every group is placed entirely in training or in validation. For CSL, one clip of each sign trains the model and clips of the same sign by other signers are the test set.

Place the prepared files in `MyDrive/SMIC/data`:

| File | Content |
|---|---|
| `isl_grouped.npz` | ISL-IEEE, 35 classes, group-disjoint split |
| `asl_grouped.npz` | ASL-IEEE, 24 classes, group-disjoint split |
| `csl_signer.npz` | CSL, 97 classes, 8 sign frames per clip, signer-disjoint |
| `csl_pool.npz` | unlabelled frames of CSL signs outside the evaluation set |

Each file stores `X` (uint8, 96 px RGB), `y`, `train_idx` and `val_idx`.

In [ ]:
def load(name):
    d = np.load(os.path.join(DATA_DIR, FILES[name] + '.npz'))
    X = torch.from_numpy(d['X']).float().div_(255)
    video = X.dim() == 5                                    # N, T, H, W, C
    if video:
        N, T = X.shape[:2]
        X = X.reshape(N * T, *X.shape[2:]).permute(0, 3, 1, 2)
        X = F.interpolate(X, size=(RES, RES), mode='area')
        X = X.reshape(N, T, 3, RES, RES)
    else:
        X = F.interpolate(X.permute(0, 3, 1, 2), size=(RES, RES), mode='area')
    tr, va = d['train_idx'], d['val_idx']
    classes = np.unique(np.concatenate([d['y'][tr], d['y'][va]]))
    remap = {int(c): i for i, c in enumerate(classes)}
    y = torch.tensor([remap.get(int(v), -1) for v in d['y']])
    return X, y, tr, va, len(classes), video


def lookup_baseline(X, y, tr, va, video):
    # 1-nearest-neighbour label copy in a 32x32 grayscale cosine descriptor.
    # Nothing is learned, so a trained model is only informative above this score.
    def desc(A):
        if video:
            A = A.mean(1)
        g = F.interpolate(A.mean(1, keepdim=True), size=(32, 32), mode='area').flatten(1)
        g = g - g.mean(1, keepdim=True)
        return g / g.norm(dim=1, keepdim=True).clamp_min(1e-8)
    j = (desc(X[va]) @ desc(X[tr]).T).argmax(1)
    return float((y[tr][j] == y[va]).float().mean())


DATA, LOOKUP = {}, {}
rows = []
for name in CORPORA:
    DATA[name] = load(name)
    X, y, tr, va, nc, video = DATA[name]
    LOOKUP[name] = lookup_baseline(X, y, tr, va, video)
    rows.append({'corpus': name.upper(), 'input': 'clip, T=%d' % X.shape[1] if video else 'image',
                 'classes': nc, 'train': len(tr), 'validation': len(va),
                 'lookup': round(LOOKUP[name], 3), 'chance': round(1 / nc, 3)})
pd.DataFrame(rows).set_index('corpus')

## 3. Models

The trunk has four blocks of two 3x3 convolutions with batch normalisation and ReLU, each followed by 2x2 max-pooling. A 64 px image leaves the trunk as a 4x4x96 map, which is read as 16 spatial tokens; a clip gives 8 temporal tokens, one pooled vector per sign frame. The projection head is used only during contrastive pretraining.

The referenced STLAT model is imported from `model.py` in the repository.

In [ ]:
from model import ReferencedSTLAT


class HandEncoder(nn.Module):
    def __init__(self, width=32, out_dim=256):
        super().__init__()
        w = width

        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                nn.MaxPool2d(2))

        self.net = nn.Sequential(block(3, w), block(w, 2 * w), block(2 * w, 4 * w),
                                 block(4 * w, 4 * w), nn.AdaptiveAvgPool2d(1))
        self.out_dim = 4 * w
        self.proj = nn.Sequential(nn.Linear(self.out_dim, 256), nn.ReLU(True),
                                  nn.Linear(256, out_dim))

    def forward(self, x, project=False):
        h = self.net(x).flatten(1)
        return F.normalize(self.proj(h), dim=-1) if project else h


def augment_batch(x, strong=True):
    # Affine jitter for every view; brightness and random grayscale for the
    # strong views used in contrastive pretraining.
    B = x.shape[0]
    ang = (torch.rand(B, device=x.device) * 2 - 1) * (0.42 if strong else 0.20)
    sc = 1 + (torch.rand(B, device=x.device) * 2 - 1) * (0.28 if strong else 0.14)
    tx = (torch.rand(B, device=x.device) * 2 - 1) * (0.22 if strong else 0.11)
    ty = (torch.rand(B, device=x.device) * 2 - 1) * (0.22 if strong else 0.11)
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    th = torch.zeros(B, 2, 3, device=x.device)
    th[:, 0, 0], th[:, 0, 1], th[:, 0, 2] = cos, -sin, tx
    th[:, 1, 0], th[:, 1, 1], th[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(th, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, padding_mode='border', align_corners=False)
    if strong:
        b = 1 + (torch.rand(B, 1, 1, 1, device=x.device) * 2 - 1) * 0.35
        x = (x * b).clamp(0, 1)
        gray = x.mean(1, keepdim=True)
        m = (torch.rand(B, 1, 1, 1, device=x.device) < 0.25).float()
        x = m * gray.expand_as(x) + (1 - m) * x
    return x


def augment_input(x, video):
    if not video:
        return augment_batch(x, strong=False)
    return augment_batch(x.flatten(0, 1), strong=False).reshape(x.shape)


class Net(nn.Module):
    # One trunk, five sequence heads. SMIC is the 'cnn' head with a pretrained trunk.
    def __init__(self, head, n_classes, video, state=None):
        super().__init__()
        enc = HandEncoder(width=WIDTH)
        if state is not None:
            enc.load_state_dict({k: v.clone() for k, v in state.items()})
        self.trunk = enc.net[:-1]                           # without global pooling
        d = enc.out_dim
        self.video, self.head = video, head
        n_tokens = 8 if video else 16
        if head == 'cnn_tf':
            layer = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True)
            self.tf = nn.TransformerEncoder(layer, 2)
            self.pos = nn.Parameter(torch.zeros(1, n_tokens, d))
        elif head == 'lstm':
            self.rnn = nn.LSTM(d, 64, 2, batch_first=True, dropout=0.2)
            d = 64
        elif head == 'stlat':
            self.st = ReferencedSTLAT(d, 64, n_classes, num_layers=4, num_heads=4,
                                      transformer_layers=2, dim_feedforward=128,
                                      dropout=0.2, use_asfg=True)
        if head != 'stlat':                                 # STLAT has its own classifier
            self.norm = nn.LayerNorm(d)
            self.fc = nn.Linear(d, n_classes)

    def tokens(self, x):
        if self.video:
            B, T = x.shape[:2]
            h = self.trunk(x.flatten(0, 1))
            return h.mean((2, 3)).reshape(B, T, -1)         # temporal tokens
        return self.trunk(x).flatten(2).transpose(1, 2)     # spatial tokens

    def forward(self, x):
        z = self.tokens(x)
        if self.head == 'stlat':
            return self.st(z)[0]
        if self.head == 'cnn_tf':
            z = self.tf(z + self.pos).mean(1)
        elif self.head == 'lstm':
            z = self.rnn(z)[0][:, -1]
        else:
            z = z.mean(1)
        return self.fc(self.norm(z))


params = {NAMES[m]: sum(p.numel() for p in Net('cnn' if m == 'smic' else m, 24, False).parameters())
          for m in MODELS}
pd.Series(params, name='parameters (ASL-IEEE)').to_frame()

## 4. Stage 1: contrastive pretraining

The loss is the normalised temperature-scaled cross-entropy of SimCLR. In a batch of N images, the two views of the same image are the only positive pair. Each pretraining run takes 20 epochs of 80 steps at batch size 128, with Adam at 1e-3 and weight decay 1e-4, from seed 0. No label is used. Trained encoders are cached in `MyDrive/SMIC/cache`.

In [ ]:
def nt_xent(z1, z2, t=TEMPERATURE):
    z = torch.cat([z1, z2], 0)
    N = z1.shape[0]
    sim = (z @ z.t()) / t
    sim.fill_diagonal_(-1e9)
    target = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, target)


def pretrain(X_uint8, steps_per_epoch=PRETRAIN_STEPS, epochs=PRETRAIN_EPOCHS,
             batch=PRETRAIN_BATCH, seed=0):
    X = torch.from_numpy(X_uint8).permute(0, 3, 1, 2).float().div_(255)
    X = F.interpolate(X, size=(RES, RES), mode='area')
    seed_all(seed)
    enc = HandEncoder(width=WIDTH).to(DEV)
    opt = torch.optim.Adam(enc.parameters(), lr=1e-3, weight_decay=1e-4)
    n, history = len(X), []
    for ep in range(epochs):
        enc.train()
        total = seen = 0
        for _ in range(steps_per_epoch):
            idx = torch.randint(0, n, (min(batch, n),))
            xb = X[idx].to(DEV)
            loss = nt_xent(enc(augment_batch(xb), project=True),
                           enc(augment_batch(xb), project=True))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.detach()) * xb.size(0)
            seen += xb.size(0)
        history.append(total / seen)
    return {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}, history


def pretraining_images(name):
    if name == 'csl':
        return np.load(os.path.join(DATA_DIR, 'csl_pool.npz'))['X']     # sign-disjoint pool
    d = np.load(os.path.join(DATA_DIR, FILES[name] + '.npz'))
    return d['X'][d['train_idx']]                                    # training split only


ENCODERS = {}
fig, ax = plt.subplots(figsize=(6, 3.2))
for name in CORPORA:
    path = os.path.join(CACHE_DIR, 'smic_encoder_%s.pt' % name)
    if os.path.exists(path):
        ENCODERS[name] = torch.load(path, map_location='cpu')
        print('%s: loaded cached encoder' % name.upper())
        continue
    images = pretraining_images(name)
    t0 = time.time()
    ENCODERS[name], history = pretrain(images)
    torch.save(ENCODERS[name], path)
    print('%s: %d unlabelled images, final loss %.3f, %.0f s'
          % (name.upper(), len(images), history[-1], time.time() - t0))
    ax.plot(range(1, len(history) + 1), history, label=name.upper())
ax.set_xlabel('epoch')
ax.set_ylabel('NT-Xent loss')
ax.set_title('Contrastive pretraining')
if ax.lines:
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Stage 2: supervised training and evaluation

Every model is trained for 40 epochs with Adam (learning rate 1e-3, weight decay 1e-3), cosine decay to zero and batch size 64 for images or 32 for clips. Accuracy, macro-F1 and macro one-vs-rest ROC-AUC are computed on the held-out split at the final epoch. Each corpus and model is run with the same five seeds, and results are written to `MyDrive/SMIC/results` after each corpus, so an interrupted session can resume.

In [ ]:
def macro_f1(pred, y):
    scores = []
    for c in np.unique(y):
        tp = ((pred == c) & (y == c)).sum()
        fp = ((pred == c) & (y != c)).sum()
        fn = ((pred != c) & (y == c)).sum()
        scores.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(scores))


def macro_auc(prob, y):
    present = np.unique(y)
    if len(present) < 2:
        return float('nan')
    p = prob[:, present] / prob[:, present].sum(1, keepdims=True)
    return float(roc_auc_score(y, p, multi_class='ovr', average='macro', labels=present))


def train_and_evaluate(name, model, seed):
    X, y, tr, va, nc, video = DATA[name]
    seed_all(seed)
    state = ENCODERS[name] if model == 'smic' else None
    net = Net('cnn' if model == 'smic' else model, nc, video, state).to(DEV)
    Xtr, ytr, Xva, yva = X[tr].to(DEV), y[tr].to(DEV), X[va].to(DEV), y[va].to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()
    bs = 32 if video else 64
    curve = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for ep in range(EPOCHS):
        net.train()
        perm = torch.randperm(len(Xtr), device=DEV)
        tl = tc = 0.0
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            out = net(augment_input(Xtr[idx], video))
            loss = crit(out, ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            tl += loss.item() * len(idx)
            tc += float((out.argmax(1) == ytr[idx]).sum())
        sched.step()
        net.eval()
        with torch.no_grad():
            logits = torch.cat([net(Xva[i:i + 128]) for i in range(0, len(Xva), 128)])
        curve['train_loss'].append(tl / len(Xtr))
        curve['train_acc'].append(tc / len(Xtr))
        curve['val_loss'].append(float(crit(logits, yva)))
        curve['val_acc'].append(float((logits.argmax(1) == yva).float().mean()))
    prob = F.softmax(logits, 1).cpu().numpy()
    yv = yva.cpu().numpy()
    pred = prob.argmax(1)
    return {'acc': float((pred == yv).mean()), 'f1': macro_f1(pred, yv),
            'auc': macro_auc(prob, yv), 'train_loss': curve['train_loss'][-1],
            'val_loss': curve['val_loss'][-1], 'curve': curve,
            'params': int(sum(p.numel() for p in net.parameters()))}

In [ ]:
RESULTS = {}
for name in CORPORA:
    path = os.path.join(RESULTS_DIR, 'bench_%s.json' % name)
    if os.path.exists(path):
        RESULTS[name] = json.load(open(path))
        print('%s: loaded saved results' % name.upper())
        continue
    RESULTS[name] = {'lookup': LOOKUP[name], 'chance': 1 / DATA[name][4], 'models': {}}
    for model in MODELS:
        t0 = time.time()
        runs = [train_and_evaluate(name, model, s) for s in SEEDS]
        RESULTS[name]['models'][model] = {
            'runs': {k: [r[k] for r in runs] for k in ('acc', 'f1', 'auc', 'train_loss', 'val_loss')},
            'params': runs[0]['params'], 'curve_seed42': runs[0]['curve']}
        accs = RESULTS[name]['models'][model]['runs']['acc']
        print('%s  %-30s acc %.3f +/- %.3f  (%.0f s)'
              % (name.upper(), NAMES[model], np.mean(accs), np.std(accs, ddof=1), time.time() - t0))
    json.dump(RESULTS[name], open(path, 'w'))

## 6. Results and statistical tests

Mean and standard deviation over the five shared seeds. The p-values are paired t-tests against STLAT over the same seeds. All five seeds share one split, so the tests measure variation between training runs, not variation from resampling the corpus.

In [ ]:
def summary_table(name):
    res = RESULTS[name]
    base = np.array(res['models']['stlat']['runs']['acc'])
    rows = []
    for model in MODELS:
        r = res['models'][model]['runs']
        acc = np.array(r['acc'])
        row = {'model': NAMES[model]}
        for key, label in (('acc', 'accuracy'), ('f1', 'macro-F1'), ('auc', 'ROC-AUC')):
            v = np.array(r[key])
            row[label] = '%.3f +/- %.3f' % (np.nanmean(v), np.nanstd(v, ddof=1))
        row['train loss'] = round(float(np.mean(r['train_loss'])), 3)
        row['val. loss'] = round(float(np.mean(r['val_loss'])), 3)
        row['parameters'] = res['models'][model]['params']
        row['p vs STLAT'] = '' if model == 'stlat' else '%.4f' % stats.ttest_rel(acc, base).pvalue
        rows.append(row)
    return pd.DataFrame(rows).set_index('model')


for name in CORPORA:
    print('%s   lookup %.3f   chance %.3f' % (name.upper(), RESULTS[name]['lookup'], RESULTS[name]['chance']))
    display(summary_table(name))

The effect of pretraining alone is measured by comparing SMIC with the CNN head, which is the same network trained from random initialisation.

In [ ]:
rows = []
for name in CORPORA:
    a = np.array(RESULTS[name]['models']['smic']['runs']['acc'])
    b = np.array(RESULTS[name]['models']['cnn']['runs']['acc'])
    c = np.array(RESULTS[name]['models']['stlat']['runs']['acc'])
    d = a - c
    rows.append({'corpus': name.upper(),
                 'SMIC - CNN': round(float((a - b).mean()), 3),
                 'p (SMIC vs CNN)': round(float(stats.ttest_rel(a, b).pvalue), 4),
                 'SMIC - STLAT': round(float(d.mean()), 3),
                 'p (SMIC vs STLAT)': float('%.2g' % stats.ttest_rel(a, c).pvalue),
                 "Cohen's d_z": round(float(d.mean() / d.std(ddof=1)), 2),
                 'seeds won': '%d/5' % int((d > 0).sum())})
pd.DataFrame(rows).set_index('corpus')

For comparison, the mean accuracies reported in the paper are stored in the repository under `results/`. Small differences are expected on different hardware, because GPU kernels are not bit-deterministic by default.

In [ ]:
rows = []
for name in CORPORA:
    paper = json.load(open(os.path.join(REPO, 'results', 'bench_%s.json' % name)))
    for model in MODELS:
        key = 'proposed' if model == 'smic' else model
        rows.append({'corpus': name.upper(), 'model': NAMES[model],
                     'this run': round(float(np.mean(RESULTS[name]['models'][model]['runs']['acc'])), 3),
                     'paper': round(paper['models'][key]['mean']['acc'], 3)})
pd.DataFrame(rows).set_index(['corpus', 'model'])

## 7. Training curves

Validation accuracy and training and validation loss per epoch for seed 42. The dotted line marks the lookup baseline.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
for j, name in enumerate(CORPORA):
    for model in MODELS:
        c = RESULTS[name]['models'][model]['curve_seed42']
        ep = range(1, len(c['val_acc']) + 1)
        axes[0, j].plot(ep, c['val_acc'], label=NAMES[model])
        axes[1, j].plot(ep, c['val_loss'], label=NAMES[model])
    axes[0, j].axhline(RESULTS[name]['lookup'], color='black', linestyle=':', linewidth=1)
    axes[0, j].set_title(name.upper())
    axes[1, j].set_xlabel('epoch')
axes[0, 0].set_ylabel('validation accuracy')
axes[1, 0].set_ylabel('validation loss')
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Proposition checks and unit tests

The three propositions in the Appendix are verified numerically over randomised instances, and the unit tests check that the models build, that seeding is reproducible and that the configuration is read correctly. Neither step needs the corpora.

In [ ]:
!cd "$REPO" && python theory.py

In [ ]:
!cd "$REPO" && python -m unittest discover -s tests -v